# Notebook 2 – Méthodes de débruitage

**PFE 2025/2026** – Segmentation d'images en présence de bruit  

---

Ce notebook compare les quatre filtres de débruitage :
1. **Filtre Gaussien** – lissage simple
2. **Filtre Médian** – robuste au bruit S&P
3. **Filtre Bilatéral** – préservation des contours
4. **Non-Local Means (NLM)** – meilleure qualité

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from pathlib import Path
from noise import add_gaussian_noise, add_salt_and_pepper_noise, compute_psnr
from denoising import gaussian_filter, median_filter, bilateral_filter, nlm_filter

print('Modules chargés ✓')

In [ ]:
# Charger image
data_dir = Path('../data/images')
imgs = list(data_dir.glob('*.png')) + list(data_dir.glob('*.jpg'))

if imgs:
    image = cv2.imread(str(imgs[0]))
else:
    image = np.zeros((256, 256, 3), dtype=np.uint8)
    cv2.circle(image, (128, 128), 80, (200, 200, 200), -1)
    cv2.rectangle(image, (20, 20), (80, 80), (30, 30, 30), -1)

# Ajouter bruit Gaussien (σ=40)
noisy_g = add_gaussian_noise(image, sigma=40)
# Ajouter bruit S&P (d=0.10)
noisy_sp = add_salt_and_pepper_noise(image, density=0.10)
print(f'Image shape: {image.shape}')

## 2.1 Comparaison sur bruit Gaussien (σ=40)

In [ ]:
def compare_filters(noisy, clean, title_prefix):
    filters = {
        'Gaussien\n(k=5)':   gaussian_filter(noisy, kernel_size=5),
        'Médian\n(k=5)':     median_filter(noisy, kernel_size=5),
        'Bilatéral':         bilateral_filter(noisy),
        'NLM':               nlm_filter(noisy),
    }
    
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    
    for ax, (name, img) in zip(axes[2:], filters.items()):
        psnr = compute_psnr(clean, img)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{name}\nPSNR={psnr:.1f}dB', fontsize=8)
        ax.axis('off')
    
    axes[0].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original', fontsize=8)
    axes[0].axis('off')
    
    psnr_noisy = compute_psnr(clean, noisy)
    axes[1].imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Bruité\nPSNR={psnr_noisy:.1f}dB', fontsize=8)
    axes[1].axis('off')
    
    fig.suptitle(title_prefix, fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig, filters

fig1, _ = compare_filters(noisy_g, image, 'Débruitage – Bruit Gaussien (σ=40)')
plt.savefig('../results/figures/debruitage_gaussien.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Comparaison sur bruit Sel-et-Poivre (d=0.10)

In [ ]:
fig2, _ = compare_filters(noisy_sp, image, 'Débruitage – Bruit Sel-et-Poivre (d=0.10)')
plt.savefig('../results/figures/debruitage_sp.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 PSNR après débruitage en fonction du niveau de bruit

In [ ]:
import pandas as pd

filter_funcs = {
    'Gaussien':   lambda img: gaussian_filter(img, kernel_size=5),
    'Médian':     lambda img: median_filter(img, kernel_size=5),
    'Bilatéral':  bilateral_filter,
    'NLM':        nlm_filter,
}

rows = []
for sigma in [10, 25, 50, 75]:
    noisy = add_gaussian_noise(image, sigma=sigma)
    for fname, func in filter_funcs.items():
        denoised = func(noisy)
        rows.append({
            'Filtre': fname, 'Niveau de bruit (σ)': sigma,
            'PSNR après débruitage': compute_psnr(image, denoised)
        })

df = pd.DataFrame(rows)
print(df.pivot(index='Niveau de bruit (σ)', columns='Filtre', values='PSNR après débruitage').round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for fname, grp in df.groupby('Filtre'):
    ax.plot(grp['Niveau de bruit (σ)'], grp['PSNR après débruitage'], marker='o', label=fname)

ax.set_xlabel('Écart-type σ (bruit Gaussien)')
ax.set_ylabel('PSNR (dB) après débruitage')
ax.set_title('Efficacité des filtres de débruitage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/psnr_filtres.png', dpi=150, bbox_inches='tight')
plt.show()